# Checkpoints

PathSim supports saving and loading simulation state via checkpoints. This allows you to pause a simulation, save its complete state to disk, and resume it later from exactly where you left off.

Checkpoints use a split format: a JSON file for metadata and structure, and an NPZ file for numerical data (block states, solver histories, etc.).

## Setup

We'll use a damped harmonic oscillator as our test system. First, let's run it continuously for 25 seconds as our reference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathsim import Simulation, Connection
from pathsim.blocks import Integrator, Amplifier, Adder, Scope

In [ ]:
# System parameters
x0, v0 = 2, 5
m, c, k = 0.8, 0.2, 1.5

def make_system():
    """Helper to create a fresh harmonic oscillator simulation."""
    I1 = Integrator(v0)
    I2 = Integrator(x0)
    A1 = Amplifier(c)
    A2 = Amplifier(k)
    A3 = Amplifier(-1/m)
    P1 = Adder()
    Sc = Scope(labels=["velocity", "position"])

    blocks = [I1, I2, A1, A2, A3, P1, Sc]
    connections = [
        Connection(I1, I2, A1, Sc), 
        Connection(I2, A2, Sc[1]),
        Connection(A1, P1), 
        Connection(A2, P1[1]), 
        Connection(P1, A3),
        Connection(A3, I1)
    ]

    sim = Simulation(blocks, connections, dt=0.01)
    return sim, Sc

## Reference Run

Run the full simulation continuously for 25 seconds. This is our ground truth.

In [ ]:
sim_ref, scope_ref = make_system()
sim_ref.run(25)

time_ref, data_ref = scope_ref.read()

## Save Checkpoint

Now let's run a second simulation, but only for the first 10 seconds. Then we save a checkpoint.

In [ ]:
sim_a, scope_a = make_system()
sim_a.run(10)

# Save checkpoint (creates checkpoint.json and checkpoint.npz)
sim_a.save_checkpoint("checkpoint")
print(f"Saved checkpoint at t = {sim_a.time:.1f}s")

## Load Checkpoint and Resume

Create an entirely new simulation with fresh block objects, load the checkpoint, and continue for the remaining 15 seconds. The key point is that the new simulation has completely different Python objects, yet the checkpoint restores the exact state by matching blocks by type and insertion order.

In [ ]:
sim_b, scope_b = make_system()

# Load checkpoint into the fresh simulation
sim_b.load_checkpoint("checkpoint")
print(f"Resumed from t = {sim_b.time:.1f}s")

# Continue the simulation for the remaining 15 seconds
sim_b.run(15)

## Compare Results

Now let's overlay the reference (continuous run) with the checkpointed run (first 10s + resumed 15s). If checkpointing works correctly, they should be identical.

In [ ]:
# Read data from both scopes
time_a, data_a = scope_a.read()
time_b, data_b = scope_b.read()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

# Position (channel 1)
ax1.plot(time_ref, data_ref[1], "k-", alpha=0.3, lw=3, label="reference (continuous)")
ax1.plot(time_a, data_a[1], "C0-", label="first half (0-10s)")
ax1.plot(time_b, data_b[1], "C1--", label="resumed (10-25s)")
ax1.axvline(10, color="gray", ls=":", alpha=0.5, label="checkpoint")
ax1.set_ylabel("position")
ax1.legend(loc="upper right", fontsize=8)

# Velocity (channel 0)
ax2.plot(time_ref, data_ref[0], "k-", alpha=0.3, lw=3, label="reference (continuous)")
ax2.plot(time_a, data_a[0], "C0-", label="first half (0-10s)")
ax2.plot(time_b, data_b[0], "C1--", label="resumed (10-25s)")
ax2.axvline(10, color="gray", ls=":", alpha=0.5)
ax2.set_ylabel("velocity")
ax2.set_xlabel("time [s]")

fig.suptitle("Checkpoint Save / Load")
fig.tight_layout()
plt.show()

The resumed simulation (dashed) seamlessly continues the reference (gray), confirming that the complete simulation state was correctly saved and restored across different Python objects.

## Checkpoint File Contents

The JSON file contains human-readable metadata about the simulation state. Let's inspect it.

In [ ]:
import json

with open("checkpoint.json") as f:
    cp = json.load(f)

print(f"PathSim version: {cp['pathsim_version']}")
print(f"Simulation time: {cp['simulation']['time']:.1f}s")
print(f"Solver: {cp['simulation']['solver']}")
print(f"Blocks saved: {len(cp['blocks'])}")
for b in cp["blocks"]:
    print(f"  {b['_key']} ({b['type']})")

Blocks are matched by type and insertion order (`Integrator_0`, `Integrator_1`, etc.), which means the checkpoint can be loaded into any simulation with the same block structure, regardless of the specific Python objects.